In [9]:
!ls

consumer_anomaly.py  consumer_filter.py  lab2.ipynb   transactions_10k.jsonl
consumer_count.py    lab1.ipynb		 producer.py


In [10]:
!pwd

/home/jovyan/notebooks/data



%%file producer.py to oznacza jezeli uruchmonie ta komorke to caly ten wynikz tej komorki zostanie zapsana do pliku producer.py

In [6]:
%%file producer.py



from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    #python jest jezykiem obiekowym wiec moge sobie jako parametr wpakowac funkcje
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}| {tx['user_id']} | {tx['timestamp']}")
    time.sleep(0.5) #jakbym nie ial tego ogranicznika czasowego to by to w jedna 1 przelecialo.

producer.flush()
producer.close()

Overwriting producer.py


#  <BR> PRACA DOMOWA <BR> #
## <BR> Napisz konsumenta wykrywającego anomalie prędkości: alert jeśli ten sam user_id wykona więcej niż 3 transakcje w ciągu 60 sekund. <BR>

In [19]:
%%file consumer_anomaly.py
from datetime import datetime
from pyspark.sql.functions import to_timestamp
from pyspark.sql.functions import col, unix_timestamp
from kafka import KafkaConsumer
import json
from collections import Counter, defaultdict


consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_counts = Counter()
user_time=defaultdict(list)
msg_count = 0

for message in consumer:
    user_counts[message.value['user_id']]+=1
    user_time[message.value['user_id']].append(message.value['timestamp'])
    for key,value in user_counts.items():
        if value >3:
            #najpozniejsza dodany czas
            t1 = datetime.fromisoformat(user_time[key][value-1])
            #najwczesnijszy
            t0 = datetime.fromisoformat(user_time[key][0])
            time = (t1 - t0).total_seconds()
            if time <=60:
                print(f"ALERT: ten sam user_id: {key}  wykonal {value} transakcje w ciągu {time} sekund (czyli mniej niz 60).")
                user_counts[key]=0
                user_time[key].clear()            

Overwriting consumer_anomaly.py


### Część 3: Konsument bezstanowy — filtrowanie

In [4]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    if message.value['amount'] >3000:
       print(f"ALERT: tranakcja wyniosla: {message.value}") 
    

Writing consumer_filter.py


In [22]:
%%file consumer_enrich.py


from kafka import KafkaProducer
import json, random, time
from datetime import datetime
from kafka import KafkaConsumer
import json

consumer =KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='transactions-risk',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)


# TWÓJ KOD
# Czytaj z 'transactions' (użyj INNEGO group_id!)
# Dodaj pole risk_level na podstawie amount
# Wypisz wzbogaconą transakcję

for message in consumer:
    tx=message.value
    
    if message.value['amount'] >3000:
        tx['risk_level']='HIGH'
        print(tx)
        #producer.send('transactions_enrich',value=tx)
    elif message.value['amount']>1000:
        tx['risk_level']='MEDIUM'
        print(tx)
       # producer.send('transactions_enrich',value=tx)
    else:
        tx['risk_level']='LOW'
        print(tx)
       # producer.send('transactions_enrich',value=tx)



        
        
    






Overwriting consumer_enrich.py


<br> Napisz konsumenta, który prowadzi bieżące zliczanie transakcji per sklep i wypisuje podsumowanie co 10 wiadomości.<br>

In [5]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter, defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0


print("PODSUMOWANIE DLA 10 WIADOMOSCI: ")
for message in consumer:
    tx=message.value
    #print(message.value)
    store_counts[message.value['store']] += 1
    total_amount[message.value['store']] += message.value['amount']
    msg_count+=1
    if msg_count==10:
        
        print(f"{'Sklep':<10} | {'Liczba':>5} | {'Suma':>10} | {'Średnia':>8}")
        for k,v in store_counts.items():

            print(f"{k :<10} | {v:>6} | {round(total_amount[k],2):>10} | {round((total_amount[k])/v,2):>8}")   
        msg_count=0
        store_counts.clear()
        total_amount.clear()
        print("---------------------------------------------")
        
        
    


Writing consumer_count.py


## <BR>Napisz konsumenta, który śledzi per kategoria: - liczbę transakcji - łączny przychód - min i max kwotę<BR>

In [47]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json
from collections import Counter, defaultdict
#Napisz konsumenta, który śledzi per kategoria: - liczbę transakcji - łączny przychód - min i max kwotę



consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='stats-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
details_amount=defaultdict(list)
msg_count = 0


print("PODSUMOWANIE DLA 10 WIADOMOSCI: ")
for message in consumer:
    store_counts[message.value['category']]+=1
    total_amount[message.value['category']]+=message.value['amount']
    details_amount[message.value['category']].append(message.value['amount'])
    msg_count+=1
    if msg_count==10:
        print(f"{'Kategoria':<15} | {'liczba transakcji':<18} | {'łączny przychód':>15} | {'min':>10} | {'max':>10}")
        for k,v in store_counts.items():

            print(f"{k :<15} | {v:<18} | {round(total_amount[k],2):>15} | {round(min(details_amount[k]),2):>10} | {round(max(details_amount[k]),2):>10}")   
        msg_count=0
        store_counts.clear()
        total_amount.clear()
        details_amount.clear()
        print("---------------------------------------------")


Overwriting consumer_stats.py
